In [ ]:
import arviz as az
import pymc as pm
import numpy as np
import os
import matplotlib.pyplot as plt
from scipy import stats
from functools import partial

In [ ]:
RANDOM_SEED = 8923
rng = np.random.default_rng(RANDOM_SEED)

Create a toy "true" background distribution, using a mixture of Normal distributions, truncated to match the range of the data.

In [ ]:
LOWER = 22
UPPER = 26.2
NBINS = 20
bkgd_n_components_true = 3
bkgd_weight_true = np.array([0.5, 0.2, 0.3])
bkgd_mu_true = np.array([23, 25, 27])
bkgd_sigma_true = np.array([2, 1.5, 2])
BKGD_NSAMP = 10000  # this is the expectation of the number of background objects in the outermost region

In [ ]:
def trunc_limits(mu, sigma):
    a = (LOWER - mu) / sigma
    b = (UPPER - mu) / sigma
    return a, b

In [ ]:
def create_mixnorm_sample(nsamp, weight, mu, sigma):
    a, b = trunc_limits(mu, sigma)
    component = rng.choice(weight.size, size=stats.poisson.rvs(nsamp), p=weight)
    samples = stats.truncnorm.rvs(a[component], b[component], mu[component], sigma[component])
    return samples

In [ ]:
bkgd_samples = create_mixnorm_sample(BKGD_NSAMP, bkgd_weight_true, bkgd_mu_true, bkgd_sigma_true)

In [ ]:
def create_density(weight, mu, sigma, n_points=1000):
    n_comp = len(weight)
    a, b = trunc_limits(mu, sigma)
    m = np.linspace(LOWER, UPPER, n_points)
    density_comp = []
    for i in range(n_comp):
        density_comp.append(stats.truncnorm.pdf(m, a[i], b[i], loc=mu[i], scale=sigma[i]) * weight[i])
    density = np.sum(density_comp, axis=0)
    return m, density, density_comp

In [ ]:
m, bkgd_true, _ = create_density(bkgd_weight_true, bkgd_mu_true, bkgd_sigma_true)

In [ ]:
plt.plot(m, bkgd_true, label="true", lw=3)
bkgd_hist, bins, _ = plt.hist(bkgd_samples, bins=NBINS, density=True, histtype="step", label=f"~{BKGD_NSAMP} samples")
plt.legend();

Model the background with a simple Gaussian mixture model. At first I tried a general model with free means and sigmas. However, this (a) takes quite a long time to sample and (b) proved difficult to robustly sample with truncated distributions. Instead, I used an overlapping set of Gauusians with fixed the means and sigmas and just allowed the heights to vary. This is similar to the approach [Blanton et al. (2008)](https://iopscience.iop.org/article/10.1086/375776) used to model the SDSS galaxy luminosity function.

In [ ]:
def create_bkgd_model(n_components=7):
    with pm.Model() as model:
        pm.Data("lower", LOWER)
        pm.Data("upper", UPPER)
        pm.Data("bkgd_n_components", n_components)
        magrange = UPPER - LOWER
        # margin = 0.5 * magrange / n_components
        margin = 0.0
        mu = pm.Data("bkgd_mu", np.linspace(LOWER - margin, UPPER + margin, n_components))
        sigma = pm.Data("bkgd_sigma", np.ones(n_components) * (magrange + 2 * margin) / n_components)
        w = pm.Dirichlet("bkgd_w", 1.1 * np.ones(n_components))
        components = pm.TruncatedNormal.dist(mu=mu, sigma=sigma, lower=LOWER, upper=UPPER)
        likelihood = pm.Mixture("bkgd", w=w, comp_dists=components, observed=bkgd_samples)
    return model

In [ ]:
bkgd_model = create_bkgd_model()

In [ ]:
if not os.path.exists("idata_bkgd.nc"):
    with bkgd_model:
        idata_bkgd = pm.sample_prior_predictive(samples=10)
        idata_bkgd.extend(pm.sample(nuts_sampler="numpyro"))
        thinned_idata_bkgd = idata_bkgd.sel(chain=[0])
        idata_bkgd.extend(pm.sample_posterior_predictive(thinned_idata_bkgd, return_inferencedata=True))
        idata_bkgd.to_netcdf("idata_bkgd.nc")
else:
    idata_bkgd = az.from_netcdf("idata_bkgd.nc")

In [ ]:
az.summary(idata_bkgd)

In [ ]:
az.plot_pair(idata_bkgd, divergences=True)
plt.tight_layout();

In [ ]:
az.plot_trace(idata_bkgd)
plt.tight_layout();

In [ ]:
def get_var_mean(var_name, n_var, df, idata, batched=True):
    if var_name in idata.constant_data:
        var = idata.constant_data[var_name].data
    else:
        template = var_name
        template += "[{}]" if batched else "{}"
        var = df.loc[[template.format(i) for i in range(n_var)], "mean"].values
    return var


def get_var_samples(var_name, n_samp, idata):
    if var_name in idata.constant_data:
        var = np.ones((n_samp, 1)) * idata.constant_data[var_name].values[None, :]
    else:
        var = idata.posterior[var_name][0].values
    return var


def hist_quantiles_from_posterior_predictive(samples, q=(0.05, 0.95)):
    histogram_samples = np.array([np.histogram(x, bins=NBINS, range=(LOWER, UPPER), density=True)[0] for x in samples])
    bin_edges = np.linspace(LOWER, UPPER, NBINS + 1)    
    hist_quantiles = np.quantile(histogram_samples, q, axis=0)
    hist_quantiles = np.concatenate((hist_quantiles, hist_quantiles[:, [-1]]), axis=-1)
    return bin_edges, hist_quantiles


def create_mean_density(name, n_components, idata):
    df = az.summary(idata)
    w = get_var_mean(f"{name}_w", n_components, df, idata)
    mu = get_var_mean(f"{name}_mu", n_components, df, idata)
    sigma = get_var_mean(f"{name}_sigma", n_components, df, idata)
    return create_density(w, mu, sigma)        


def dens_quantiles_from_posterior(name, n_draw, idata, q=(0.05, 0.95)):
    w = get_var_samples(f"{name}_w", n_draw, idata)
    mu = get_var_samples(f"{name}_mu", n_draw, idata)
    sigma = get_var_samples(f"{name}_sigma", n_draw, idata)
    dens_samples = np.array([create_density(w[i], mu[i], sigma[i])[1]
                             for i in range(len(w))])
    dens_quantiles = np.quantile(dens_samples, q, axis=0)
    return dens_quantiles


def plot_background(idata, true_density):
    n_components = int(idata.constant_data.bkgd_n_components.data)
    try:
        bkgd_samples = idata.observed_data.bkgd.data
    except AttributeError:
        bkgd_samples = None 

    m, bkgd_mean, bkgd_mean_comp = create_mean_density("bkgd", n_components, idata)

    plt.plot(m, true_density, lw=2, color="firebrick", label="true background")
    plt.plot(m, bkgd_mean, lw=2, color="steelblue", label="mean posterior")
    for comp in bkgd_mean_comp:
        plt.plot(m, comp, lw=2, color="steelblue", ls=":")
    if bkgd_samples is not None:
        bins, (hist_05, hist_95) = hist_quantiles_from_posterior_predictive(idata.posterior_predictive.bkgd[0])
        plt.hist(bkgd_samples, bins=bins, color="darkorange", density=True, histtype="step", lw=2, label=f"~{BKGD_NSAMP} observed samples")
        plt.fill_between(bins, hist_05, hist_95, step="post", color="lightgreen", lw=0, label=f"95% interval on sampled")
    dens_05, dens_95 = dens_quantiles_from_posterior("bkgd", len(idata.posterior.draw), idata)
    plt.fill_between(m, dens_05, dens_95, step="mid", color="skyblue", lw=0, label="95% interval on mean")
    plt.legend(loc="lower center")
    plt.ylabel("normalised density")
    plt.xlabel("magnitude")
    plt.xlim(xmin=LOWER, xmax=UPPER)

In [ ]:
plot_background(idata_bkgd, true_density=bkgd_true)

You can see that the model is flexible enough such that the mean posterior closely follows the histogram of the observed samples. This may suggest that it is "overfitting" the observations. However, this doesn't particularly matter, as we will not be making any use of the mean posterior. Instead, what we care about is the posterior distribution. You can see that the credible interval on the mean (created from the posterior samples) encompasses the "true" distribution and the credible interval on the the histogram of the samples reflects the variation in the observed histogram from the "truth".

Now create a distribution for the GCs.

In [ ]:
N_REGIONS = 5
REGION_AREAS = np.array([1.0, 0.2, 0.1, 0.05, 0.02])
gclf_n_components_true = 2
gclf_mu_true = np.array([26.3, 26.6])
gclf_sigma_true = np.array([1.5, 0.6])
#gclf_mu_true = np.array([25.0, 26.6])
#gclf_sigma_true = np.array([1.0, 0.6])
# The surface density of each GC population in each region, relative to the background population
gclf_reldens_true = np.array([[0.02, 0.01], [0.5, 0.2], [2.0, 0.5], [7.0, 1.5], [25.0, 5.0]])

In [ ]:
ref_val_plot_pair = {"gclf_mu[0]": gclf_mu_true[0], "gclf_mu[1]": gclf_mu_true[1],
                     "gclf_sigma[0]": gclf_sigma_true[0], "gclf_sigma[1]": gclf_sigma_true[1]}
ref_val_plot_posterior = {k: [{"ref_val": v}] for k, v in ref_val_plot_pair.items()}
gclf_reldens_sum = gclf_reldens_true.sum(axis=-1)
gclf_comp0_w = gclf_reldens_true[:, 0] / gclf_reldens_sum
ref_val_plot_pair.update({f"gclf_comp0_w_region {i}": w for i, w in enumerate(gclf_comp0_w)})
ref_val_plot_posterior.update({"gclf_comp0_w_region": [{f"gclf_comp0_w_region_dim_0": i ,"ref_val": w} for i, w in enumerate(gclf_comp0_w)]})
gclf_w = gclf_reldens_sum / (gclf_reldens_sum + 1)
ref_val_plot_pair.update({f"gclf_weight_region {i}": w for i, w in enumerate(gclf_w)})
ref_val_plot_posterior.update({"gclf_weight_region": [{f"gclf_weight_region_dim_0": i ,"ref_val": w} for i, w in enumerate(gclf_w)]})

In [ ]:
gclf_samples = []
bkgd_samples = []
for r in range(N_REGIONS):
    bkgd_n = int(BKGD_NSAMP * REGION_AREAS[r])
    gclf_n = int(bkgd_n * gclf_reldens_true[r].sum())
    gclf_weight = gclf_reldens_true[r] / gclf_reldens_true[r].sum()
    gclf_samples.append(create_mixnorm_sample(gclf_n, gclf_weight, gclf_mu_true, gclf_sigma_true))
    bkgd_samples.append(create_mixnorm_sample(bkgd_n, bkgd_weight_true, bkgd_mu_true, bkgd_sigma_true))

In [ ]:
gclf_true = []
gclf_true_comp = []
for r in range(N_REGIONS):
    gclf_count = (BKGD_NSAMP * REGION_AREAS[r] * gclf_reldens_true[r]).astype(int)
    gclf_count = rng.poisson(gclf_count)
    _, true, comp = create_density(gclf_count, gclf_mu_true, gclf_sigma_true)
    gclf_true.append(true)
    gclf_true_comp.append(comp)
gclf_true = np.array(gclf_true)
gclf_true_comp = np.array(gclf_true_comp)

In [ ]:
def plot_samples_in_regions(samples, true, comp=None):
    gclf_hist, bins, _ = plt.hist(samples, bins=NBINS, range=(LOWER, UPPER), density=False, histtype="step")
    plt.gca().set_prop_cycle(None)
    for r in range(N_REGIONS):
        plt.plot(m, true[r] * (UPPER - LOWER) / NBINS)
    if comp is not None:
        n_components = len(comp[0])
        for i in range(n_components):
            plt.gca().set_prop_cycle(None)
            for r in range(N_REGIONS):
                plt.plot(m, comp[r][i] * (UPPER - LOWER) / NBINS, ls=":")
    plt.xlim(xmin=LOWER, xmax=UPPER)
    plt.xlabel("mag")
    plt.ylabel("count")

In [ ]:
plot_samples_in_regions(gclf_samples, gclf_true, gclf_true_comp)

In [ ]:
bkgd_true_regions = []
bkgd_true_regions_comp = []
for r in range(N_REGIONS):
    bkgd_n = int(BKGD_NSAMP * REGION_AREAS[r])
    bkgd_count = (BKGD_NSAMP * REGION_AREAS[r] * bkgd_weight_true).astype(int)
    bkgd_count = rng.poisson(bkgd_count)
    _, true, comp = create_density(bkgd_count, bkgd_mu_true, bkgd_sigma_true)
    bkgd_true_regions.append(true)
    bkgd_true_regions_comp.append(comp)

In [ ]:
plot_samples_in_regions(bkgd_samples, bkgd_true_regions)

In [ ]:
full_true = [gclf_true[r] + bkgd_true_regions[r] for r in range(N_REGIONS)]

In [ ]:
full_samples = [np.concatenate((gclf_samples[r], bkgd_samples[r])) for r in range(N_REGIONS)]

In [ ]:
plot_samples_in_regions(full_samples, full_true)

In [ ]:
def create_model_full(n_bkgd_comp=7, n_gclf_comp=2, gclf_mu0=None, gclf_mu1=None, gclf_sigma0=None, gclf_sigma1=None,
                      truncated=True):
    if truncated:
        NormalDist = partial(pm.TruncatedNormal.dist, lower=LOWER, upper=UPPER)
    else:
        NormalDist = pm.Normal.dist
    with pm.Model() as model:
        # the background
        pm.Data("lower", LOWER)
        pm.Data("upper", UPPER)
        pm.Data("bkgd_n_components", n_bkgd_comp)
        magrange = UPPER - LOWER
        # margin = 0.5 * magrange / n_bkgd_comp
        margin = 0.0
        bkgd_mu = pm.Data("bkgd_mu", np.linspace(LOWER - margin, UPPER + margin, n_bkgd_comp))
        bkgd_sigma = pm.Data("bkgd_sigma", np.ones(n_bkgd_comp) * (magrange + 2 * margin) / n_bkgd_comp)
        bkgd_w = pm.Dirichlet("bkgd_w", 1.1 * np.ones(n_bkgd_comp))
        bkgd_components = [NormalDist(mu=bkgd_mu[i], sigma=bkgd_sigma[i]) for i in range(n_bkgd_comp)]

        # the signal
        pm.Data("gclf_n_components", n_gclf_comp)
        if gclf_mu0 is not None:
            mu0 = pm.Data("gclf_mu0", gclf_mu0)
        else:
            mu0 = pm.Normal("gclf_mu[0]", mu=26.0, sigma=1.0, initval=26.0)
        if gclf_sigma0 is not None:
            sigma0 = pm.Data("gclf_sigma[0]", gclf_sigma0)
        else:
            sigma0 = pm.Gamma("gclf_sigma[0]", mu=1.5, sigma=0.5)
        if n_gclf_comp > 1:
            if gclf_mu1 is not None:
                mu1 = pm.Data("gclf_mu[1]", gclf_mu1)
            else:
                mu1 = pm.Normal("gclf_mu[1]", mu=26.5, sigma=1.0, initval=26.5)
            if gclf_sigma1 is not None:
                sigma1 = pm.Data("gclf_sigma[1]", gclf_sigma1)
            else:
                sigma1 = pm.Gamma("gclf_sigma[1]", mu=0.5, sigma=0.5)

        if n_gclf_comp > 1:
            if gclf_mu0 is None or gclf_mu1 is None:
                mu_constraint = mu0 < mu1
                pm.Potential("mu_constraint", pm.math.log(pm.math.switch(mu_constraint, 1, 0)))
            if gclf_sigma0 is None or gclf_sigma1 is None:
                sigma_constraint = sigma1 < sigma0
                pm.Potential("sigma_constraint", pm.math.log(pm.math.switch(sigma_constraint, 1, 0)))

        gclf_comp0 = NormalDist(mu=mu0, sigma=sigma0)
        if n_gclf_comp > 1:
            gclf_comp1 = NormalDist(mu=mu1, sigma=sigma1)

        if n_gclf_comp > 1:
            region_components = [gclf_comp0, gclf_comp1] + bkgd_components
        else:
            region_components = [gclf_comp0] + bkgd_components

        # Using Beta to mimic a Uniform distibution as there seems to be a bug with Uniform, at least for shape > 1
        region_gclf_weight = pm.Beta("gclf_weight_region", alpha=1, beta=1, shape=N_REGIONS,
                                     initval=np.linspace(0.1, 0.9, N_REGIONS))
        # impose ordering on region weights
        #region_constraint = pm.math.min(region_gclf_weight[1:] - region_gclf_weight[:-1]) > 0
        #pm.Potential("region_constraint", pm.math.log(pm.math.switch(region_constraint, 1, 0.1)))

        if n_gclf_comp > 1:
            region_gclf_comp0_w = pm.Beta("gclf_comp0_w_region", alpha=1, beta=1, shape=N_REGIONS)
        for r in range(N_REGIONS):
            if n_gclf_comp > 1:
                region_w = pm.math.concatenate((region_gclf_weight[None, r] * region_gclf_comp0_w[r],
                                                region_gclf_weight[None, r] * (1 - region_gclf_comp0_w[r]),
                                                (1 - region_gclf_weight[r]) * bkgd_w))
            else:
                region_w = pm.math.concatenate((region_gclf_weight[None, r],
                                               (1 - region_gclf_weight[r]) * bkgd_w))
            region_like = pm.Mixture(f"full_region{r}", w=region_w, comp_dists=region_components, observed=full_samples[r])
    return model

In [ ]:
def sample_model(model):
    with model:
        idata = pm.sample_prior_predictive()
        idata.extend(pm.sample(init="adapt_diag", nuts_sampler="numpyro"))
        thinned_idata = idata.sel(chain=[0])
        idata.extend(pm.sample_posterior_predictive(thinned_idata, return_inferencedata=True))
    return idata

In [ ]:
def plot_model(idata, priors=False):
    ax = az.plot_ppc(idata, group="prior")
    plt.suptitle("Prior Predictive Check")
    plt.tight_layout()
    plot_pair_kwargs = dict(filter_vars="like",
                            kind="kde", kde_kwargs=dict(hdi_probs=[0.68, 0.95],
                            contour_kwargs=dict(alpha=[0, 1, 1, 1]),
                            contourf_kwargs=dict(cmap=plt.cm.Blues)),
                            reference_values=ref_val_plot_pair,
                            reference_values_kwargs=dict(markersize=10, color="orange"))
    az.plot_trace(idata)
    plt.suptitle("MCMC Traces")
    plt.tight_layout()
    try:
        az.plot_posterior(idata, var_names=["gclf_sigma", "gclf_mu"], filter_vars="like",
                          ref_val=ref_val_plot_posterior)
        plt.suptitle("Posterior – GCLF Distribution Parameters")
        plt.tight_layout()
    except ValueError:
        pass
    az.plot_posterior(idata, var_names="gclf_weight", filter_vars="like",
                      ref_val=ref_val_plot_posterior)
    plt.suptitle("Posterior – GCLF Weights Vs Background")
    plt.tight_layout()
    try:
        az.plot_posterior(idata, var_names="gclf_comp", filter_vars="like",
                          ref_val=ref_val_plot_posterior)
        plt.suptitle("Posterior – GCLF Component Weights")
        plt.tight_layout()
    except ValueError:
        pass
    try:
        if priors:
            az.plot_pair(idata, var_names=["gclf_sigma", "gclf_mu"], group="prior", **plot_pair_kwargs)    
            plt.suptitle("Prior Pair Plot – GCLF Distribution Parameters")
            plt.tight_layout()
        az.plot_pair(idata, var_names=["gclf_sigma", "gclf_mu"], **plot_pair_kwargs)    
        plt.suptitle("Posterior Pair Plot – GCLF Distribution Parameters")
        plt.tight_layout()
    except ValueError:
        pass
    if priors:
        az.plot_pair(idata, var_names="gclf_weight", group="prior", **plot_pair_kwargs)
        plt.suptitle("Prior Pair Plot – GCLF Weights Vs Background")
        plt.tight_layout()
    az.plot_pair(idata, var_names="gclf_weight", **plot_pair_kwargs)
    plt.suptitle("Posterior Pair Plot – GCLF Weights Vs Background")
    plt.tight_layout()
    try:
        if priors:
            az.plot_pair(idata, var_names="gclf_comp", group="prior", **plot_pair_kwargs)
            plt.suptitle("Prior Pair Plot – GCLF Component Weights")
            plt.tight_layout()
        az.plot_pair(idata, var_names="gclf_comp", **plot_pair_kwargs)
        plt.suptitle("Posterior Pair Plot – GCLF Component Weights")
        plt.tight_layout()
    except ValueError:
        pass
    ax = az.plot_ppc(idata)
    plt.suptitle("Posterior Predictive Check")
    plt.tight_layout()
    plt.figure()
    plot_background(idata, true_density=bkgd_true)
    #plot_full TBD

To be done:

## Sample and explore models

In [ ]:
model = {}
idata = {}

### Two-component GCLF model with fixed means and sigmas

In [ ]:
name = "full_2_comp_all_fixed"
model[name] = create_model_full(gclf_mu0=gclf_mu_true[0], gclf_mu1=gclf_mu_true[1],
                                gclf_sigma0=gclf_sigma_true[0], gclf_sigma1=gclf_sigma_true[1])

In [ ]:
pm.model_to_graphviz(model[name], var_names=["full_region0"])

In [ ]:
if not os.path.exists(f"{name}.nc"):
    idata[name] = sample_model(model[name])
    idata[name].to_netcdf(f"{name}.nc")
else:
    idata[name] = az.from_netcdf(f"{name}.nc")

In [ ]:
az.summary(idata[name])

In [ ]:
plot_model(idata[name])

### Two-component GCLF model with fixed means and free sigmas

In [ ]:
name = "full_2comp_mean_fixed"
model[name] = create_model_full(gclf_mu0=gclf_mu_true[0], gclf_mu1=gclf_mu_true[1])

In [ ]:
pm.model_to_graphviz(model[name], var_names=["full_region0"])

In [ ]:
if not os.path.exists(f"{name}.nc"):
    idata[name] = sample_model(model[name])
    idata[name].to_netcdf(f"{name}.nc")
else:
    idata[name] = az.from_netcdf(f"{name}.nc")

In [ ]:
az.summary(idata[name])

In [ ]:
plot_model(idata[name])

### One-component GCLF model with fixed mean and free sigma

In [ ]:
name = "full_1comp_mean_fixed"
model[name] = create_model_full(n_gclf_comp=1, gclf_mu0=gclf_mu_true[0])

In [ ]:
pm.model_to_graphviz(model[name], var_names=["full_region0"])

In [ ]:
if not os.path.exists(f"{name}.nc"):
    idata[name] = sample_model(model[name])
    idata[name].to_netcdf(f"{name}.nc")
else:
    idata[name] = az.from_netcdf(f"{name}.nc")

In [ ]:
az.summary(idata[name])

In [ ]:
plot_model(idata[name])

### One-component GCLF model with free mean and sigma

In [ ]:
name = "full_1comp_all_free"
model[name] = create_model_full(n_gclf_comp=1)

In [ ]:
pm.model_to_graphviz(model[name], var_names=["full_region0"])

In [ ]:
if not os.path.exists(f"{name}.nc"):
    idata[name] = sample_model(model[name])
    idata[name].to_netcdf(f"{name}.nc")
else:
    idata[name] = az.from_netcdf(f"{name}.nc")

In [ ]:
az.summary(idata[name])

In [ ]:
plot_model(idata[name])

### Two-component GCLF model with free means and sigmas

In [ ]:
name = "full_2comp_all_free"
model[name] = create_model_full()

In [ ]:
pm.model_to_graphviz(model[name], var_names=["full_region0"])

In [ ]:
if not os.path.exists(f"{name}.nc"):
    idata[name] = sample_model(model[name])
    idata[name].to_netcdf(f"{name}.nc")
else:
    idata[name] = az.from_netcdf(f"{name}.nc")

In [ ]:
az.summary(idata[name])

In [ ]:
plot_model(idata[name])

## Model comparison

In [ ]:
for name in model:
    with model[name]:
        pm.compute_log_likelihood(idata[name])

In [ ]:
df_comp_loo = az.compare(idata, var_name="full_region0")
df_comp_loo

In [ ]:
az.plot_bf(idata["full_2comp_mean_fixed"], var_name="gclf_sigma[0]", ref_val=gclf_sigma_true[0]);

To be done: